# MahaMonsoon-X — ERA5 Data Audit

Fast first-pass audit for `data/era5/era5_1940_2026.grib`.

**Performance design:** scan GRIB headers for all messages, but decode numerical grid values for only a few messages per variable. No full DataFrame conversion, no CSV export of the gridded data, and no full-dataset load into memory.

GRIB is a multidimensional grid format, not a conventional table. Here, “rows/columns” means message-level records/metadata fields and, separately, grid rows (`Nj`) and columns (`Ni`). Distribution plots are sampled, not full-dataset distributions. Missing counts are exact only when the GRIB message exposes usable missing-value metadata.

## 1. Configuration and dependencies

Run from the project root. Change `GRIB_PATH` if needed.

In [ ]:
from pathlib import Path
from collections import defaultdict
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from eccodes import (
    codes_get, codes_get_values, codes_grib_new_from_file, codes_release
)

GRIB_PATH = Path("../../data/era5/era5_monthly_1940_2026.grib")
SAMPLE_MESSAGES_PER_VARIABLE = 3
MAX_PLOT_POINTS_PER_VARIABLE = 20_000
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

if not GRIB_PATH.exists():
    raise FileNotFoundError(f"File not found: {GRIB_PATH.resolve()}")

print(f"File: {GRIB_PATH.resolve()}")
print(f"Size: {GRIB_PATH.stat().st_size / (1024**2):.2f} MiB")

## 2. Fast metadata scan

Reads message headers without unpacking every grid's numerical values. This still visits each GRIB message, but is much lighter than decoding the entire dataset.

In [ ]:
def key(handle, name, default=None):
    try:
        return codes_get(handle, name)
    except Exception:
        return default

records = []
sample_numbers = defaultdict(list)
message_number = 0

with GRIB_PATH.open("rb") as stream:
    while True:
        h = codes_grib_new_from_file(stream)
        if h is None:
            break
        try:
            message_number += 1
            var = str(key(h, "shortName", "unknown"))
            records.append({
                "message_number": message_number,
                "short_name": var,
                "name": key(h, "name"),
                "units": key(h, "units"),
                "data_date": key(h, "dataDate"),
                "validity_date": key(h, "validityDate"),
                "data_time": key(h, "dataTime"),
                "validity_time": key(h, "validityTime"),
                "step_type": key(h, "stepType"),
                "step_range": key(h, "stepRange"),
                "type_of_level": key(h, "typeOfLevel"),
                "level": key(h, "level"),
                "packing_type": key(h, "packingType"),
                "bitmap_present": key(h, "bitmapPresent"),
                "number_of_missing": key(h, "numberOfMissing"),
                "number_of_points": key(h, "numberOfPoints"),
                "Ni": key(h, "Ni"),
                "Nj": key(h, "Nj"),
                "grid_type": key(h, "gridType"),
                "data_type": key(h, "dataType"),
                "missing_value": key(h, "missingValue"),
                "lat_first": key(h, "latitudeOfFirstGridPointInDegrees"),
                "lat_last": key(h, "latitudeOfLastGridPointInDegrees"),
                "lon_first": key(h, "longitudeOfFirstGridPointInDegrees"),
                "lon_last": key(h, "longitudeOfLastGridPointInDegrees"),
            })
            if len(sample_numbers[var]) < SAMPLE_MESSAGES_PER_VARIABLE:
                sample_numbers[var].append(message_number)
        finally:
            codes_release(h)
        if message_number % 2000 == 0:
            print(f"Scanned {message_number:,} message headers...")

metadata = pd.DataFrame(records)
if metadata.empty:
    raise RuntimeError("No GRIB messages found.")

print("\n========== COUNTS ==========")
print(f"Message-level rows: {len(metadata):,}")
print(f"Metadata columns: {len(metadata.columns):,}")
print(f"Variables (short names): {metadata['short_name'].nunique():,}")
print(f"Distinct validity dates overall: {metadata['validity_date'].nunique():,}")
print("\nMetadata DataFrame shape:", metadata.shape)
print("\nMetadata column data types:")
display(metadata.dtypes.astype(str).rename("dtype").to_frame())

print("\nGrid dimensions (Nj = grid rows, Ni = grid columns):")
display(
    metadata.groupby(["Ni", "Nj", "number_of_points"], dropna=False)
    .size().reset_index(name="message_count")
    .sort_values("message_count", ascending=False)
)

print("\nMessage count and date coverage by variable:")
display(
    metadata.groupby("short_name")
    .agg(messages=("message_number", "count"),
         first_validity_date=("validity_date", "min"),
         last_validity_date=("validity_date", "max"),
         unique_validity_dates=("validity_date", "nunique"),
         units=("units", "first"), name=("name", "first"))
    .reset_index().sort_values("short_name")
)

## 3. Missing-value metadata and variable characteristics

`numberOfMissing` is used when ecCodes exposes a valid value. Missing metadata is reported as unknown, not incorrectly treated as zero. `packing_type` describes GRIB encoding; it is not the decoded NumPy dtype.

In [ ]:
for c in ["number_of_missing", "number_of_points", "Ni", "Nj"]:
    metadata[c + "_num"] = pd.to_numeric(metadata[c], errors="coerce")

def joined_unique(series, limit=4):
    vals = list(dict.fromkeys(str(x) for x in series.dropna()))
    return ", ".join(vals[:limit]) + (", ..." if len(vals) > limit else "") if vals else "Unknown"

audit_rows = []
for var, g in metadata.groupby("short_name", sort=True):
    miss = g["number_of_missing_num"]
    pts = g["number_of_points_num"]
    exact = miss.notna().all() and pts.notna().all()
    total_missing = int(miss.sum()) if exact else np.nan
    total_points = int(pts.sum()) if pts.notna().all() else np.nan
    pct = 100 * total_missing / total_points if exact and total_points else np.nan
    audit_rows.append({
        "short_name": var,
        "name": joined_unique(g["name"]),
        "units": joined_unique(g["units"]),
        "messages": len(g),
        "Ni_grid_columns": joined_unique(g["Ni"]),
        "Nj_grid_rows": joined_unique(g["Nj"]),
        "points_per_message": joined_unique(g["number_of_points"]),
        "packing_type": joined_unique(g["packing_type"]),
        "step_type": joined_unique(g["step_type"]),
        "step_range": joined_unique(g["step_range"]),
        "bitmap_present": joined_unique(g["bitmap_present"]),
        "missing_count_if_available": total_missing,
        "total_grid_points": total_points,
        "missing_percent_if_available": pct,
        "missing_count_source": "GRIB metadata" if exact else "Unavailable/incomplete",
        "first_validity_date": g["validity_date"].min(),
        "last_validity_date": g["validity_date"].max(),
        "unique_validity_dates": g["validity_date"].nunique(),
    })

audit = pd.DataFrame(audit_rows)
display(audit)

## 4. Decode a small sample only

At most three messages per variable are decoded. These samples provide actual NumPy dtypes, sampled non-finite counts, ranges, and plot data. This is intentionally not an exact full-data numerical audit.

In [ ]:
selected = {n for ns in sample_numbers.values() for n in ns}
samples = defaultdict(list)
stats = []
decoded_dtypes = {}
current = 0

with GRIB_PATH.open("rb") as stream:
    while True:
        h = codes_grib_new_from_file(stream)
        if h is None:
            break
        current += 1
        if current not in selected:
            codes_release(h)
            continue
        try:
            var = str(key(h, "shortName", "unknown"))
            arr = np.asarray(codes_get_values(h))
            decoded_dtypes[var] = str(arr.dtype)
            arr = arr.astype(np.float64, copy=False)
            finite = arr[np.isfinite(arr)]
            nonfinite = int(arr.size - finite.size)
            used = finite
            if used.size > MAX_PLOT_POINTS_PER_VARIABLE:
                used = rng.choice(used, MAX_PLOT_POINTS_PER_VARIABLE, replace=False)
            samples[var].append(used.copy())
            stats.append({
                "short_name": var,
                "message_number": current,
                "decoded_numpy_dtype": str(arr.dtype),
                "grid_values_in_message": arr.size,
                "non_finite_values_in_sampled_message": nonfinite,
                "finite_values_retained": used.size,
                "sample_min": float(used.min()) if used.size else np.nan,
                "sample_max": float(used.max()) if used.size else np.nan,
                "sample_mean": float(used.mean()) if used.size else np.nan,
                "sample_median": float(np.median(used)) if used.size else np.nan,
                "sample_std": float(used.std()) if used.size else np.nan,
                "sample_zero_percent": float(np.mean(used == 0) * 100) if used.size else np.nan,
            })
        finally:
            codes_release(h)

sample_stats = pd.DataFrame(stats)
display(sample_stats)

print("Decoded NumPy dtype per variable (sampled messages only):")
display(pd.DataFrame([
    {"short_name": v, "decoded_numpy_dtype": dtype}
    for v, dtype in sorted(decoded_dtypes.items())
]))

## 5. Candidate imputation strategy

This is a preliminary recommendation, not an automatic transformation. Zero rainfall is a valid value and must not be used as a default replacement for missing rainfall.

In [ ]:
def imputation_candidate(row):
    p = row["missing_percent_if_available"]
    if pd.isna(p):
        return "Exact count unavailable from metadata; inspect bitmap/sample or run a deliberate full value scan."
    if p == 0:
        return "No missing values reported by metadata; verify sample and physical ranges."
    if p <= 1:
        return "Locate gaps; spatial interpolation may suit isolated cells if physically justified."
    if p <= 5:
        return "Map missingness; compare interpolation with excluding affected cells/times."
    return "Investigate masks/source coverage; avoid blind imputation and consider excluding affected cells/periods."

strategy = audit[[
    "short_name", "name", "units", "missing_percent_if_available", "missing_count_source"
]].copy()
strategy["candidate_imputation_strategy"] = strategy.apply(imputation_candidate, axis=1)

if not sample_stats.empty:
    sample_summary = sample_stats.groupby("short_name").agg(
        sampled_messages=("message_number", "count"),
        sampled_non_finite_values=("non_finite_values_in_sampled_message", "sum"),
        sampled_min=("sample_min", "min"),
        sampled_max=("sample_max", "max"),
    ).reset_index()
    strategy = strategy.merge(sample_summary, on="short_name", how="left")

display(strategy)
print("Do not impute until missingness pattern, physical ranges, and variable units are reviewed.")

## 6. Box plots for every variable

Separate panels are used because variables have different units and scales. Plots use only sampled values. Outlier points are hidden for readability; the box-and-whisker statistics still reflect the sampled values.

In [ ]:
plot_vars = [
    v for v in sorted(samples)
    if any(a.size for a in samples[v])
]

if not plot_vars:
    print("No finite sampled values available.")
else:
    cols = 3
    rows = math.ceil(len(plot_vars) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3.4 * rows), squeeze=False)
    for ax, var in zip(axes.flat, plot_vars):
        vals = np.concatenate([a for a in samples[var] if a.size])
        ax.boxplot(vals, showfliers=False)
        ax.set_title(f"{var} — sampled values")
        ax.set_ylabel("Native GRIB units")
        ax.set_xticks([])
        ax.grid(True, axis="y", alpha=0.25)
    for ax in axes.flat[len(plot_vars):]:
        ax.set_visible(False)
    fig.suptitle("ERA5 variable box plots (sampled values)", y=1.01)
    fig.tight_layout()
    plt.show()

## 7. Distribution plot with a smooth density curve

Each variable gets a histogram and, when SciPy is available and the data have enough variation, a kernel density estimate (KDE) curve. Otherwise the histogram is still displayed. These are sample distributions in native units.

In [ ]:
try:
    from scipy.stats import gaussian_kde
    HAVE_SCIPY = True
except ImportError:
    HAVE_SCIPY = False
    print("SciPy not available: histogram will be shown without KDE curves.")

if not plot_vars:
    print("No finite sampled values available.")
else:
    cols = 2
    rows = math.ceil(len(plot_vars) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 3.8 * rows), squeeze=False)
    for ax, var in zip(axes.flat, plot_vars):
        vals = np.concatenate([a for a in samples[var] if a.size])
        vals = vals[np.isfinite(vals)]
        if not vals.size:
            ax.set_title(f"{var} — no finite sample")
            continue
        ax.hist(vals, bins=40, density=True, alpha=0.55)
        if HAVE_SCIPY and vals.size >= 3 and np.unique(vals).size > 1:
            kde_vals = vals
            if kde_vals.size > 5000:
                kde_vals = rng.choice(kde_vals, size=5000, replace=False)
            try:
                kde = gaussian_kde(kde_vals)
                lo, hi = np.quantile(kde_vals, [0.005, 0.995])
                if np.isfinite(lo) and np.isfinite(hi) and hi > lo:
                    x = np.linspace(lo, hi, 250)
                    ax.plot(x, kde(x), linewidth=1.8)
            except (ValueError, np.linalg.LinAlgError):
                pass
        ax.set_title(f"{var} — sampled distribution")
        ax.set_xlabel("Native GRIB units")
        ax.set_ylabel("Density")
        ax.grid(True, alpha=0.2)
    for ax in axes.flat[len(plot_vars):]:
        ax.set_visible(False)
    fig.suptitle("ERA5 variable distributions (sampled values)", y=1.01)
    fig.tight_layout()
    plt.show()

## 8. Timestamp and duplicate-date audit

This reports duplicate validity dates and the range for each variable. Do not blindly shift month-end dates to month-start: accumulated and flux variables can have different temporal conventions that must be reconciled using their ERA5 product semantics.

In [ ]:
time_rows = []
for var, g in metadata.groupby("short_name", sort=True):
    d = pd.to_numeric(g["validity_date"], errors="coerce").dropna().astype(int)
    counts = d.value_counts()
    time_rows.append({
        "short_name": var,
        "messages": len(g),
        "unique_validity_dates": int(d.nunique()),
        "duplicate_date_messages": int((counts - 1).clip(lower=0).sum()),
        "first_validity_date": int(d.min()) if len(d) else np.nan,
        "last_validity_date": int(d.max()) if len(d) else np.nan,
        "step_types": joined_unique(g["step_type"]),
        "step_ranges": joined_unique(g["step_range"]),
    })
temporal_audit = pd.DataFrame(time_rows)
display(temporal_audit)

## 9. Optional: save compact audit reports

Only small metadata tables are saved. The grid values are never exported to CSV.

In [ ]:
SAVE_REPORTS = True  # Set to True to save compact audit tables

if SAVE_REPORTS:
    out = Path("../artifacts/audit/era5_audit_report")
    out.mkdir(parents=True, exist_ok=True)
    metadata.to_csv(out / "grib_message_metadata.csv", index=False)
    audit.to_csv(out / "variable_audit.csv", index=False)
    sample_stats.to_csv(out / "sample_value_statistics.csv", index=False)
    strategy.to_csv(out / "imputation_candidates.csv", index=False)
    temporal_audit.to_csv(out / "temporal_audit.csv", index=False)
    print(f"Saved small audit reports under {out.resolve()}")
else:
    print("Report saving is off. Set SAVE_REPORTS = True to save compact audit tables.")

## 10. Review checklist before preprocessing

- All expected variables are present.
- Grid geometry is consistent.
- Validity dates and monthly coverage make sense.
- Units and `step_type` are understood, especially for precipitation and flux variables.
- Missingness is understood rather than blindly filled.
- Sampled ranges are physically plausible.
- A Maharashtra boundary mask is available if predictions must represent Maharashtra rather than the full extraction bounding box.

This notebook intentionally avoids a complete numerical pass. If exact missing counts are unavailable in GRIB metadata, an exact per-cell count requires a separate, deliberate full-value scan.